# Camera Calibration and 6DoF Object Pose

> **Advanced · 3D geometry**


## Why this matters

Calibration connects pixel coordinates to a physical camera model; 6DoF pose uses that model to locate a known object. They belong together as one geometric contract.

**Where it appears:** Measurement, robotics, augmented reality, camera undistortion, and pose-aware inspection.


## Learning Objectives

- Understand why camera calibration is required before any metric 3D vision task
- Estimate intrinsic parameters and distortion coefficients from a checkerboard pattern
- Undistort images using the computed calibration
- Estimate 3D object pose (rotation + translation) from 2D-3D point correspondences
- Use solvePnP correctly, including the RANSAC-robust variant
- Project and draw a 3D coordinate axis onto an image to verify pose visually


## Prerequisites

06 Drawing and Geometric Transformations; comfortable linear-algebra intuition

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.calibrateCamera`, `cv2.undistort`, `cv2.solvePnP`, `cv2.projectPoints`, Rodrigues vectors

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Camera Calibration

Real camera lenses introduce distortion (mainly radial and tangential),
and every camera has different intrinsic parameters (focal length,
principal point). Camera calibration recovers these from multiple views
of a known pattern (checkerboard), enabling **undistortion** (removing
lens warping) and providing the intrinsics required for any downstream
metric task: stereo vision, pose estimation, AR overlay placement.
Skipping calibration and assuming a pinhole model is a common source of
silent 3D-vision errors.


### Pose Estimation

Pose estimation (Perspective-n-Point, PnP) recovers a camera's rotation
and translation relative to an object, given known 3D points on the
object and their corresponding observed 2D image points, plus the camera
intrinsics from calibration (notebook 29). `cv2.solvePnP` is the direct
solver; `cv2.solvePnPRansac` additionally rejects outlier correspondences,
which real point detectors (feature matching, landmark detection) will
inevitably produce some of.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Camera Calibration


### 1. Simulating checkerboard views for calibration

Real calibration needs 10-20 photos of a physical checkerboard from different angles. Here we render a synthetic checkerboard under several simulated perspective distortions to demonstrate the full calibration API without hardware.


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from cv_utils import load_real_image, get_real_data, show_grid

PATTERN_SIZE = (7, 5)  # inner corners, not squares

# Load real-world calibration images (different views of the checkerboard)
calib_dir = get_real_data("images/calibration", "")
views = []
for img_path in sorted(calib_dir.glob("*.png")):
    img = cv2.imread(str(img_path))
    if img is not None:
        views.append(img)

# Show the first 5 views
show_grid([(f"view {i}", v) for i, v in enumerate(views[:5])])

### 2. Running calibration

Find checkerboard corners in each view, build the matching 3D object points (assumed planar, Z=0), and call `cv2.calibrateCamera` to recover intrinsics + distortion.


In [ ]:
def find_corners(views: list) -> tuple[list, list]:
    objp = np.zeros((PATTERN_SIZE[0] * PATTERN_SIZE[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0 : PATTERN_SIZE[0], 0 : PATTERN_SIZE[1]].T.reshape(-1, 2)

    obj_points, img_points, used_views = [], [], []
    for v in views:
        gray = cv2.cvtColor(v, cv2.COLOR_BGR2GRAY)
        found, corners = cv2.findChessboardCorners(gray, PATTERN_SIZE)
        if found:
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
            corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
            obj_points.append(objp)
            img_points.append(corners)
            used_views.append(v)
    return obj_points, img_points, used_views


obj_points, img_points, used_views = find_corners(views)
print(f"Successfully found the checkerboard in {len(used_views)}/{len(views)} views")

if len(used_views) >= 3:
    image_size = used_views[0].shape[1::-1]
    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        obj_points, img_points, image_size, None, None
    )
    print(
        "Reprojection RMS error:",
        round(ret, 4),
        "(lower is better; <1.0 is generally good)",
    )
    print("Camera matrix:\n", np.round(camera_matrix, 2))
    print("Distortion coefficients:", np.round(dist_coeffs.ravel(), 4))
else:
    print(
        "Not enough successful detections to calibrate -- need >= 3 views in practice, ideally 10-20."
    )

### 3. Undistorting an image with the calibration result

Apply the recovered camera matrix and distortion coefficients to correct lens distortion in a new image from the same (simulated) camera.


In [ ]:
if len(used_views) >= 3:
    test_view = used_views[0]
    h, w = test_view.shape[:2]
    new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix, dist_coeffs, (w, h), 1, (w, h)
    )
    undistorted = cv2.undistort(
        test_view, camera_matrix, dist_coeffs, None, new_camera_matrix
    )
    show_grid([("original view", test_view), ("undistorted", undistorted)])
else:
    print("Skipping undistortion demo -- calibration did not succeed above.")

## Part 2: Pose Estimation


### 1. Defining a known 3D object and synthetic 2D observations

Use the corners of a known-size square marker as 3D object points, and project them through a chosen, ground-truth pose to synthesize 2D observations -- so the result of `solvePnP` can be checked against a known correct answer.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid

# 3D object points: corners of a 100mm square marker, centered at origin, in its own coordinate frame
object_points = np.array(
    [
        [-50, -50, 0],
        [50, -50, 0],
        [50, 50, 0],
        [-50, 50, 0],
    ],
    dtype=np.float32,
)

camera_matrix = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]], dtype=np.float32)
dist_coeffs = np.zeros(
    5, dtype=np.float32
)  # assume already-undistorted images (notebook 29)

true_rvec = np.array(
    [[0.2], [0.4], [0.1]], dtype=np.float32
)  # ground-truth rotation (Rodrigues)
true_tvec = np.array(
    [[10], [5], [400]], dtype=np.float32
)  # ground-truth translation (mm)

observed_2d, _ = cv2.projectPoints(
    object_points, true_rvec, true_tvec, camera_matrix, dist_coeffs
)
observed_2d = observed_2d.reshape(-1, 2)
print("Synthesized 2D marker corner observations:\n", np.round(observed_2d, 1))

### 2. Solving for pose with solvePnP

Recover rotation/translation from the 2D-3D correspondences and confirm the solver's answer matches the ground truth used to synthesize the data.


In [ ]:
success, rvec, tvec = cv2.solvePnP(
    object_points, observed_2d, camera_matrix, dist_coeffs
)
print("solvePnP succeeded:", success)
print("Recovered rvec:\n", np.round(rvec, 4), "\n(true rvec was)\n", true_rvec)
print("Recovered tvec:\n", np.round(tvec, 4), "\n(true tvec was)\n", true_tvec)

rotation_error = np.linalg.norm(rvec - true_rvec)
translation_error = np.linalg.norm(tvec - true_tvec)
print(
    f"\nRotation vector error: {rotation_error:.6f}  Translation error: {translation_error:.6f} mm"
)

### 3. Visualizing pose with a projected coordinate axis

Draw the object's local X/Y/Z axes projected into the image using the recovered pose -- the standard visual sanity check used in AR marker tracking.


In [ ]:
def draw_pose_axes(
    image: np.ndarray, rvec, tvec, camera_matrix, dist_coeffs, axis_length=60
) -> np.ndarray:
    axis_points = np.float32(
        [[0, 0, 0], [axis_length, 0, 0], [0, axis_length, 0], [0, 0, -axis_length]]
    )
    projected, _ = cv2.projectPoints(
        axis_points, rvec, tvec, camera_matrix, dist_coeffs
    )
    projected = projected.reshape(-1, 2).astype(int)
    origin = tuple(projected[0])
    canvas = image.copy()
    cv2.line(canvas, origin, tuple(projected[1]), (0, 0, 255), 3)  # X = red
    cv2.line(canvas, origin, tuple(projected[2]), (0, 255, 0), 3)  # Y = green
    cv2.line(canvas, origin, tuple(projected[3]), (255, 0, 0), 3)  # Z = blue
    return canvas


canvas = np.full((480, 640, 3), 230, dtype=np.uint8)
for pt in observed_2d.astype(int):
    cv2.circle(canvas, tuple(pt), 5, (0, 165, 255), -1)
canvas = draw_pose_axes(canvas, rvec, tvec, camera_matrix, dist_coeffs)
show_grid([("marker corners + recovered pose axes", canvas)], cols=1)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Camera Calibration: Visualizing Camera Poset Extrinsics in 3D

After calibrating a camera, the rotation (`rvecs`) and translation (`tvecs`) vectors define where the camera was located relative to the checkerboard grid during each snapshot. Here, we parse these pose transformations and display them.


In [ ]:
# Mock rotation and translation vectors for 3 views
rvecs = [
    np.array([[0.1], [0.0], [0.0]]),
    np.array([[-0.2], [0.1], [0.0]]),
    np.array([[0.05], [-0.15], [0.0]]),
]
tvecs = [
    np.array([[0.0], [0.0], [5.0]]),
    np.array([[1.0], [-0.5], [6.0]]),
    np.array([[-1.0], [0.5], [4.5]]),
]


def get_camera_positions(rvecs, tvecs) -> list[np.ndarray]:
    cam_positions = []
    for rvec, tvec in zip(rvecs, tvecs):
        # Convert rotation vector to 3x3 matrix
        R, _ = cv2.Rodrigues(rvec)
        # Position in world coordinates: C = -R^T * T
        C = -R.T @ tvec
        cam_positions.append(C.flatten())
    return cam_positions


positions = get_camera_positions(rvecs, tvecs)
for i, pos in enumerate(positions):
    print(
        f"View {i + 1} estimated 3D position vector: X={pos[0]:.2f}, Y={pos[1]:.2f}, Z={pos[2]:.2f}"
    )

### Mini Project — Pose Estimation: Augmented Reality Wireframe Cube Projection

Pose estimation (`solvePnP`) yields rotation and translation vectors. We use these poses to project 3D coordinate boxes (such as a wireframe cube) onto a target flat surface, creating an augmented reality overlay.


In [ ]:
# Define 3D coordinates of a cube structure in target space (centered on board)
cube_3d = np.array(
    [
        [0, 0, 0],
        [1, 0, 0],
        [1, 1, 0],
        [0, 1, 0],  # Base coordinates
        [0, 0, -1],
        [1, 0, -1],
        [1, 1, -1],
        [0, 1, -1],  # Ceiling coordinates
    ],
    dtype=np.float32,
)

# Mock camera matrix and distortion parameters
K = np.array([[500, 0, 160], [0, 500, 120], [0, 0, 1]], dtype=np.float32)
dist = np.zeros((4, 1))

# Mock pose values (rotation and translation offsets)
rvec = np.array([[0.1], [0.2], [0.0]])
tvec = np.array([[0.0], [0.0], [5.0]])

# Project 3D cube vertices to 2D screen coordinates
pts_2d, _ = cv2.projectPoints(cube_3d, rvec, tvec, K, dist)
pts_2d = pts_2d.reshape(-1, 2).astype(np.int32)

# Draw wireframe lines on canvas
canvas = np.zeros((240, 320, 3), dtype=np.uint8)

# Connect base and ceiling loops
for i in range(4):
    cv2.line(canvas, tuple(pts_2d[i]), tuple(pts_2d[(i + 1) % 4]), (0, 255, 0), 2)
    cv2.line(
        canvas, tuple(pts_2d[i + 4]), tuple(pts_2d[((i + 1) % 4) + 4]), (0, 255, 0), 2
    )
    # Connect upright columns
    cv2.line(canvas, tuple(pts_2d[i]), tuple(pts_2d[i + 4]), (0, 255, 0), 2)

print("AR cube projection complete.")
show(canvas, "Projected AR Wireframe Cube")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Camera Calibration
1. Increase the number of simulated views to 10+ and observe whether reprojection error improves.
2. Save `camera_matrix` and `dist_coeffs` to a `.npz` file with `np.savez`, then reload them in a fresh cell.
3. Explain, in markdown, why at least 3 non-coplanar-looking (differently tilted) views are required for calibration to be well-posed.

Use the empty cell below to work through them.


#### Solutions — Camera Calibration

In [ ]:
# Solution 1: Reprojection error vs view count
# Explanation: Increasing the number of calibration views from 5 to 15+ provides the optimizer
# with a wider variety of perspective constraints, reducing the impact of pixel measurement
# noise. This results in a lower, more stable reprojection error and higher calibration precision.


In [ ]:
# Solution 2: Save and reload calibration outputs
def save_calibration_data(filepath: str, mtx: np.ndarray, dist: np.ndarray) -> None:
    """Save camera calibration matrices to a .npz file."""
    np.savez(filepath, camera_matrix=mtx, dist_coeffs=dist)


def load_calibration_data(filepath: str) -> tuple[np.ndarray, np.ndarray]:
    """Load calibration matrices from a .npz file."""
    data = np.load(filepath)
    return data["camera_matrix"], data["dist_coeffs"]

In [ ]:
# Solution 3: Why 3 non-coplanar tilted views are required
# Camera calibration optimizes for focal length, principal point coordinates, and distortion coefficients.
# If all calibration checkerboards are parallel (coplanar) to the sensor, the optimizer cannot
# separate focal length (scaling) from depth translation (Z offset), creating mathematical ambiguity.
# Tilting the checkerboards breaks this translation-scaling ambiguity, making calibration well-posed.


### Exercises — Pose Estimation
1. Add small random noise to `observed_2d` and compare `solvePnP` vs `solvePnPRansac` recovery accuracy.
2. Deliberately corrupt one of the four 2D correspondences (simulating a bad detection) and show `solvePnPRansac` still recovers a good pose while plain `solvePnP` does not.
3. Extend `object_points` to 6+ non-planar points and re-run -- note PnP is generally better-conditioned with more, non-coplanar points.

Use the empty cell below to work through them.


#### Solutions — Pose Estimation

In [ ]:
# Solution 1: solvePnP vs solvePnPRansac noise comparison
# Explanation: Adding noise to observed_2d coordinates perturbs the geometric equations.
# Plain `solvePnP` uses direct optimization, so noise directly impacts the estimated rotation
# and translation. `solvePnPRansac` fits coordinate combinations iteratively to find the best
# model, remaining robust to minor noise.


In [ ]:
# Solution 2: Outlier recovery check on corrupted coordinate
def test_outlier_robustness() -> None:
    """Verify RANSAC pose recovery when one point is corrupted."""
    obj_pts = np.array([[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0]], dtype=np.float32)
    obs_pts = obj_pts[:, :2].copy()

    # Corrupt last measurement coordinate
    obs_pts[3] += [10.0, 10.0]

    K = np.array([[500, 0, 160], [0, 500, 120], [0, 0, 1]], dtype=np.float32)
    dist = np.zeros(4)

    # Plain solvePnP fails or returns high error
    success_pnp, r_pnp, t_pnp = cv2.solvePnP(obj_pts, obs_pts, K, dist)
    # solvePnPRansac flags and rejects the corrupted point outlier
    success_ran, r_ran, t_ran, inliers = cv2.solvePnPRansac(obj_pts, obs_pts, K, dist)

    print("solvePnP status:", success_pnp)
    print(
        "solvePnPRansac inliers found count:",
        len(inliers) if inliers is not None else 0,
    )

In [ ]:
# Solution 3: Why PnP is better-conditioned with 6+ points
# A minimum of 3 (or 4 for unique solution) points is required for PnP. However, utilizing
# more coordinates (6+) provides redundant geometric equations. The optimizer can distribute
# and average out noise across these measurements, reducing error bounds and improving numerical conditioning.

test_outlier_robustness()

## Summary

You can calibrate a camera, interpret reprojection error, estimate an object pose from 2D–3D correspondences, and draw a pose axis.

- **Best Practices:** Capture calibration images across the whole field of view, retain calibration artifacts, use consistent world units, and check reprojection error on held-out views.
- **Common Pitfalls:** Calibrating from nearly identical views, mixing coordinate units, ignoring lens distortion, and confusing human-pose landmarks with 6DoF object pose.